# LinkedIn Data Cleaning & Preprocessing

This notebook cleans and prepares the LinkedIn job postings dataset (and related tables) for loading into a relational database.

**Sections**
1. Setup & Imports
2. Data Audit (Profiling)
3. Cleaning & Transformation (per table)
4. Integration & Validation
5. Export & QA Logging

**Expected input files** (in `DATA_DIR`):
- `postings.csv`, `companies.csv`, `skills.csv`, `job_skills.csv`, `industries.csv`, `job_industries.csv`, `benefits.csv`, `company_industries.csv`, `employee_counts.csv`, `company_specialities.csv`, `salaries.csv`


In [17]:
# --- 1. SETUP & IMPORTS ---
import pandas as pd
import numpy as np
import re
from pathlib import Path
from datetime import datetime

pd.set_option('display.max_columns', 200)
pd.set_option('display.width', 180)

# --- PATH CONFIGURATION ---
DATA_DIR = Path('.')  # main project folder (DB_LINKEDIN_POSTING)
OUTPUT_DIR = DATA_DIR / 'cleaned'
OUTPUT_DIR.mkdir(exist_ok=True)

# --- SAFE CSV READER ---
def read_csv_safe(path, **kwargs):
    """Read CSV safely, handling encoding errors automatically."""
    try:
        return pd.read_csv(path, **kwargs)
    except UnicodeDecodeError:
        return pd.read_csv(path, encoding='latin-1', **kwargs)

# --- FLEXIBLE FILE LOADER ---
def maybe_read(name):
    """
    Try to locate the file in DATA_DIR or any of its subfolders.
    Example: postings.csv inside ./jobs/ or ./companies/
    """
    for sub in [DATA_DIR] + list(DATA_DIR.glob('*')):
        f = sub / name
        if f.exists():
            print(f" Found {name} in {sub}")
            return read_csv_safe(f)
    print(f"[WARN]  {name} not found in {DATA_DIR} or its subfolders.")
    return None

# --- LOAD DATASETS ---
postings = maybe_read('postings.csv')
companies = maybe_read('companies.csv')
skills = maybe_read('skills.csv')
job_skills = maybe_read('job_skills.csv')
industries = maybe_read('industries.csv')
job_industries = maybe_read('job_industries.csv')
benefits = maybe_read('benefits.csv')
company_industries = maybe_read('company_industries.csv')
employee_counts = maybe_read('employee_counts.csv')
company_specialities = maybe_read('company_specialities.csv')
salaries = maybe_read('salaries.csv')

print('\n Dataset load summary:')
for n, df in [
    ('postings', postings), ('companies', companies), ('skills', skills), ('job_skills', job_skills),
    ('industries', industries), ('job_industries', job_industries), ('benefits', benefits),
    ('company_industries', company_industries), ('employee_counts', employee_counts),
    ('company_specialities', company_specialities), ('salaries', salaries)
]:
    status = ' OK' if df is not None else ' MISSING'
    print(f" - {n:<22}: {status}")


 Found postings.csv in .
 Found companies.csv in companies
 Found skills.csv in mappings
 Found job_skills.csv in jobs
 Found industries.csv in mappings
 Found job_industries.csv in jobs
 Found benefits.csv in jobs
 Found company_industries.csv in companies
 Found employee_counts.csv in companies
 Found company_specialities.csv in companies
 Found salaries.csv in jobs

 Dataset load summary:
 - postings              :  OK
 - companies             :  OK
 - skills                :  OK
 - job_skills            :  OK
 - industries            :  OK
 - job_industries        :  OK
 - benefits              :  OK
 - company_industries    :  OK
 - employee_counts       :  OK
 - company_specialities  :  OK
 - salaries              :  OK


In [18]:
# --- 2. DATA AUDIT (Profiling) ---
def audit_df(df: pd.DataFrame, name: str) -> pd.DataFrame:
    if df is None:
        print(f"[WARN] {name} is missing; skipping audit.")
        return pd.DataFrame()
    summary = pd.DataFrame({
        'dataset': name,
        'column': df.columns,
        'dtype': df.dtypes.astype(str).values,
        'null_count': df.isna().sum().values,
        'null_pct': (df.isna().sum() / len(df) * 100).round(2).values,
        'unique_count': [df[c].nunique(dropna=False) for c in df.columns]
    })
    summary.to_csv(OUTPUT_DIR / f'audit_{name}.csv', index=False)
    print(f"Audit for {name} -> {OUTPUT_DIR / f'audit_{name}.csv'} (shape={df.shape}, dups={df.duplicated().sum()})")
    return summary

all_audits = []
for nm, df in [('postings', postings), ('companies', companies), ('skills', skills),
               ('job_skills', job_skills), ('industries', industries), ('job_industries', job_industries),
               ('benefits', benefits), ('company_industries', company_industries),
               ('employee_counts', employee_counts), ('salaries', salaries)]:
    aud = audit_df(df, nm)
    if not aud.empty:
        all_audits.append(aud)

if all_audits:
    audit_master = pd.concat(all_audits, ignore_index=True)
    audit_master.to_csv(OUTPUT_DIR / 'audit_master.csv', index=False)
    audit_master.head()


Audit for postings -> cleaned\audit_postings.csv (shape=(123849, 31), dups=0)
Audit for companies -> cleaned\audit_companies.csv (shape=(24473, 10), dups=0)
Audit for skills -> cleaned\audit_skills.csv (shape=(35, 2), dups=0)
Audit for job_skills -> cleaned\audit_job_skills.csv (shape=(213768, 2), dups=0)
Audit for industries -> cleaned\audit_industries.csv (shape=(422, 2), dups=0)
Audit for job_industries -> cleaned\audit_job_industries.csv (shape=(164808, 2), dups=0)
Audit for benefits -> cleaned\audit_benefits.csv (shape=(67943, 3), dups=0)
Audit for company_industries -> cleaned\audit_company_industries.csv (shape=(24375, 2), dups=0)
Audit for employee_counts -> cleaned\audit_employee_counts.csv (shape=(35787, 4), dups=0)
Audit for salaries -> cleaned\audit_salaries.csv (shape=(40785, 8), dups=0)


In [19]:
# --- Helper functions for cleaning ---
def clean_text(x):
    if pd.isna(x):
        return x
    s = str(x)
    # remove non-ascii (emojis), lowercase, trim extra whitespace
    s = s.encode('ascii', 'ignore').decode()
    s = s.lower().strip()
    s = re.sub(r'\s+', ' ', s)
    # common standardizations
    s = re.sub(r'\bsr\.?\b', 'senior', s)
    s = re.sub(r'\bjr\.?\b', 'junior', s)
    return s

# def parse_datetime_series(s, unit=None):
#     # tries numeric epoch or string datetime
#     s2 = pd.to_datetime(s, errors='coerce', unit=unit) if unit else pd.to_datetime(s, errors='coerce')
#     # if too many NaT and we didn't try unit, try 's' unit
#     if s2.isna().mean() > 0.9 and unit is None:
#         s2 = pd.to_datetime(s, errors='coerce', unit='s')
#     return s2

def parse_datetime_series(s):
    s = pd.to_numeric(s, errors='coerce')
    def convert(x):
        if pd.isna(x):
            return pd.NaT
        x = int(x)
        # milliseconds (13 digits)
        if x > 1e12:
            return pd.to_datetime(x, unit='ms', errors='coerce')
        # seconds (10 digits)
        elif x > 1e9:
            return pd.to_datetime(x, unit='s', errors='coerce')
        # fallback: normal datetime string
        return pd.to_datetime(x, errors='coerce')
    return s.apply(convert)

def remove_outliers_3sigma(series):
    if series is None or series.isna().all():
        return series
    x = series.astype(float)
    mu, sigma = x.mean(), x.std(ddof=0)
    if sigma == 0 or np.isnan(sigma):
        return series
    return series.where((x >= mu - 3*sigma) & (x <= mu + 3*sigma))
# --- 3A. CLEAN: postings ---
if postings is not None:
    # Remove exact duplicates on typical identity fields if present
    key_cols = [c for c in ['title','company_id','location','listed_time','date_posted'] if c in postings.columns]
    if key_cols:
        before = len(postings)
        postings = postings.drop_duplicates(subset=key_cols)
        print(f"postings: removed {before - len(postings)} duplicate rows based on {key_cols}")

    # Drop rows missing essential fields
    essential = [c for c in ['title','company_id'] if c in postings.columns]
    if essential:
        before = len(postings)
        postings = postings.dropna(subset=essential)
        print(f"postings: dropped {before - len(postings)} rows missing essentials {essential}")

    # Text normalization for key text columns if present
    text_cols = [c for c in ['title','description','formatted_experience_level','location','work_type','formatted_work_type'] if c in postings.columns]
    for col in text_cols:
        postings[col] = postings[col].apply(clean_text)

    # Parse listed_time / date_posted
    for time_col in ['listed_time','date_posted','original_listed_time','closed_time','expiry']:
        if time_col in postings.columns:
            postings[time_col] = parse_datetime_series(postings[time_col])
    if 'listed_time' in postings.columns:
        postings['year'] = postings['listed_time'].dt.year

    # Format all date columns as mm-dd-yyyy strings
    for time_col in ['listed_time','date_posted','original_listed_time','closed_time','expiry']:
        if time_col in postings.columns:
            postings[time_col] = postings[time_col].dt.strftime('%m-%d-%Y')

    # Derive remote flag
    if 'formatted_work_type' in postings.columns and 'remote_allowed' not in postings.columns:
        postings['remote_allowed'] = postings['formatted_work_type'].str.contains('remote', case=False, na=False)

    # Basic numeric sanitization for salary-like fields if they exist
    for col in ['max_salary','med_salary','min_salary','applies','views','years_experience']:
        if col in postings.columns:
            postings[col] = pd.to_numeric(postings[col], errors='coerce')

    # Remove impossible salaries (<= 0)
    for col in ['max_salary','med_salary','min_salary']:
        if col in postings.columns:
            before = len(postings)
            postings.loc[postings[col] <= 0, col] = np.nan
            print(f"postings: set non-positive {col} to NaN (affected {postings[col].isna().sum()} total NaNs)")

    # Optional: outlier trimming on salary columns
    for col in ['max_salary','med_salary','min_salary']:
        if col in postings.columns:
            postings[col] = remove_outliers_3sigma(postings[col])

    # --- REMOVE SALARY COLUMNS FROM POSTINGS FOR DB NORMALIZATION ---
    salary_cols = ['min_salary','med_salary','max_salary','currency','compensation_type']
    postings = postings.drop(columns=[c for c in salary_cols if c in postings.columns], errors='ignore')

    # Standardize country/state if present (lightweight example)
    # Expectation: location might be "City, State, Country" or similar; here we just trim whitespace
    if 'location' in postings.columns:
        postings['location'] = postings['location'].str.replace(' ,', ',', regex=False).str.replace(', ', ', ', regex=False)

    print('postings cleaning complete. Shape:', postings.shape)
else:
    print('[WARN] postings.csv not found; skipping postings cleaning.')


postings: removed 2324 duplicate rows based on ['title', 'company_id', 'location', 'listed_time']
postings: dropped 1717 rows missing essentials ['title', 'company_id']
postings: set non-positive max_salary to NaN (affected 90749 total NaNs)
postings: set non-positive med_salary to NaN (affected 113747 total NaNs)
postings: set non-positive min_salary to NaN (affected 90749 total NaNs)
postings cleaning complete. Shape: (119808, 27)


In [20]:
# --- 3A + 3B. CLEAN: location fields + meaningful company size generation ---

import numpy as np
import pandas as pd
import random
import re
import unicodedata

# ----------------------------------------------------
# COMMON HELPERS
# ----------------------------------------------------

INVALID_VALUES = {"0", "00", "000", "-", "oo", "na", "n/a", "none", "", "nan"}

def clean_field(x):
    if pd.isna(x):
        return np.nan
    x = str(x).strip().lower()
    if x in INVALID_VALUES:
        return np.nan
    return x

def assign_size_from_range(low, high):
    return int(np.random.randint(low, high + 1))

def contains(text, keywords):
    text = str(text).lower()
    return any(k in text for k in keywords)

# -------------------------------
# State / Country / City helpers
# -------------------------------

# 1) US state map (normalized: no spaces in keys)
US_STATE_MAP_RAW = {
    "alabama":"AL","alaska":"AK","arizona":"AZ","arkansas":"AR","california":"CA",
    "colorado":"CO","connecticut":"CT","delaware":"DE","florida":"FL","georgia":"GA",
    "hawaii":"HI","idaho":"ID","illinois":"IL","indiana":"IN","iowa":"IA",
    "kansas":"KS","kentucky":"KY","louisiana":"LA","maine":"ME","maryland":"MD",
    "massachusetts":"MA","michigan":"MI","minnesota":"MN","mississippi":"MS",
    "missouri":"MO","montana":"MT","nebraska":"NE","nevada":"NV","new hampshire":"NH",
    "new jersey":"NJ","new mexico":"NM","new york":"NY","north carolina":"NC",
    "north dakota":"ND","ohio":"OH","oklahoma":"OK","oregon":"OR","pennsylvania":"PA",
    "rhode island":"RI","south carolina":"SC","south dakota":"SD","tennessee":"TN",
    "texas":"TX","utah":"UT","vermont":"VT","virginia":"VA","washington":"WA",
    "west virginia":"WV","wisconsin":"WI","wyoming":"WY"
}
US_STATE_MAP = {k.replace(" ", ""): v for k, v in US_STATE_MAP_RAW.items()}
VALID_US_STATES = set(US_STATE_MAP.values())

# 2) common country codes -> full name
COUNTRY_MAP = {
    "us": "United States",
    "usa": "United States",
    "united states": "United States",
    "gb": "United Kingdom",
    "uk": "United Kingdom",
    "de": "Germany",
    "ie": "Ireland",
    "ca": "Canada",
    "fr": "France",
    "se": "Sweden",
    "fi": "Finland",
    "nl": "Netherlands",
    "ch": "Switzerland",
    "in": "India",
    "dk": "Denmark",
    "au": "Australia",
    "be": "Belgium",
    "cn": "China",
    "es": "Spain",
    "sg": "Singapore",
    "it": "Italy",
    "jp": "Japan",
    "za": "South Africa",
    "br": "Brazil",
    "mx": "Mexico",
    "no": "Norway",
    "nz": "New Zealand"
}

# 3) special-character stripping
def clean_special_chars_city(x):
    if pd.isna(x):
        return np.nan
    x = re.sub(r"[^a-zA-Z ,.-]", "", str(x))
    return x.strip().title() if x.strip() else np.nan

def clean_special_chars_state(x):
    if pd.isna(x):
        return np.nan
    x = re.sub(r"[^a-zA-Z]", "", str(x))
    return x.strip().upper() if x.strip() else np.nan

def clean_special_chars_country(x):
    if pd.isna(x):
        return np.nan
    x = re.sub(r"[^a-zA-Z ]", "", str(x))
    return x.strip().title() if x.strip() else np.nan

# ----------------------------------------------------
# REGION ABBREVIATION LOGIC (2–3 LETTER CODES)
# ----------------------------------------------------

KNOWN_REGION_ABBR = {
    # Canada
    "ontario": "ON", "quebec": "QC", "britishcolumbia": "BC", "alberta": "AB",
    "manitoba": "MB", "saskatchewan": "SK", "newbrunswick": "NB",

    # Australia
    "newsouthwales": "NSW", "newsouthwalesnsw": "NSW", "nsw": "NSW",
    "victoria": "VIC", "qld": "QLD", "queensland": "QLD",
    "westernaustralia": "WA", "westernaustraliaau": "WA",
    "southaustralia": "SA",

    # UK
    "england": "ENG", "scotland": "SCT", "wales": "WLS", "northernireland": "NI",
    "greaterlondon": "GLN", "london": "LON", "middlesex": "MDX",
    "westyorkshire": "WYK", "eastflanders": "EFL", "westflanders": "WFL",

    # Germany
    "bavaria": "BY", "bayern": "BY",
    "badenwrttemberg": "BW", "badenwrtemberg": "BW",
    "hessen": "HE", "rhinelandpalatinate": "RP", "rheinlandpfalz": "RP",
    "northrhinewestphalia": "NRW", "northrhinewestphalia": "NRW",
    "nordrheinwestfalen": "NRW", "schleswigholstein": "SH",

    # France
    "iledefrance": "IDF", "ledefrance": "IDF", "idf": "IDF",
    "hautsdefrance": "HDF", "auvergnerhnealpes": "ARA",
    "provencealpescotedazur": "PACA", "occitanie": "OCC",

    # Switzerland
    "zurich": "ZH", "zrich": "ZH", "vaud": "VD", "geneva": "GE",
    "genve": "GE", "baselstadt": "BS", "baselcountry": "BL",
    "schwyz": "SZ", "schaffhausen": "SH",

    # Netherlands / Belgium
    "noordholland": "NH", "northholland": "NH",
    "zuidholland": "ZH", "southholland": "ZH",
    "noordbrabant": "NB", "brabant": "NB",
    "oostvlaanderen": "OVL", "westvlaanderen": "WVL",
    "flemishregion": "VLG",

    # Spain
    "catalonia": "CAT", "galicia": "GAL",
    "communityofmadrid": "MAD", "comunidaddemadrid": "MAD",

    # India
    "karnataka": "KA", "tamilnadu": "TN", "telangana": "TS",
    "uttarpradesh": "UP", "maharashtra": "MH", "maharastra": "MH",
    "gujarat": "GJ", "andhrapradesh": "AP", "madhyapradesh": "MP",

    # Italy
    "lombardy": "LOM", "lombardia": "LOM",
    "tuscany": "TUS", "roma": "ROM", "rome": "ROM",

    # Nordics misc
    "stockholm": "STH", "stockholmcounty": "STC",
    "oslo": "OSL", "helsinki": "HEL", "copenhagen": "CPH",
    "uusimaa": "UUS",

    # Generic
    "gauteng": "GAU",
    "hongkong": "HKG",
    "shanghai": "SHA",
    "guangdong": "GDG",
    "tokyo": "TOK",
}

def strip_accents(s):
    return ''.join(
        c for c in unicodedata.normalize('NFD', s)
        if unicodedata.category(c) != 'Mn'
    )

def auto_abbrev(region):
    """Return a 2–3 letter uppercase abbreviation for any region-like string."""
    if pd.isna(region) or not isinstance(region, str) or not region.strip():
        return np.nan

    r = strip_accents(region.lower().strip())
    r = r.replace(" ", "").replace("-", "")

    # 1) Known mapping
    if r in KNOWN_REGION_ABBR:
        return KNOWN_REGION_ABBR[r]

    # 2) If it's clearly a US state name (no space version)
    if r in US_STATE_MAP:
        return US_STATE_MAP[r]

    # 3) Long blob like "southernfinland" -> first 3 letters
    if len(r) >= 10:
        return r[:3].upper()

    # 4) Otherwise, first 2–3 letters
    if len(r) <= 2:
        return r.upper()
    return r[:3].upper()

def normalize_state(s):
    """Unify state field:
       - US states -> official 2-letter code
       - Others   -> 2–3 letter region abbreviation
    """
    if pd.isna(s):
        return np.nan

    s_raw = str(s)
    s_clean = strip_accents(s_raw.lower().strip())
    s_nospace = s_clean.replace(" ", "").replace("-", "")

    # Already a US code
    if len(s_nospace) == 2 and s_nospace.isalpha() and s_nospace.upper() in VALID_US_STATES:
        return s_nospace.upper()

    # US full name (no spaces)
    if s_nospace in US_STATE_MAP:
        return US_STATE_MAP[s_nospace]

    # Otherwise, treat as generic region
    return auto_abbrev(s_raw)

def normalize_country(c):
    if pd.isna(c):
        return np.nan
    c_str = str(c).strip()
    low = c_str.lower()
    if low in COUNTRY_MAP:
        return COUNTRY_MAP[low]
    if len(c_str) == 2 and c_str.isalpha():
        return c_str.upper()
    return c_str.title()

# -------------------------------
# Imputation helpers
# -------------------------------

def pick_address(city):
    streets = [
        "Market Street", "Innovation Drive", "Main Street", "Park Avenue",
        "Tech Park Road", "Central Avenue", "Broadway"
    ]
    city_str = city if isinstance(city, str) and city else "City"
    return f"{random.randint(100,9999)} {random.choice(streets)}, {city_str}"

ZIP_MAP = {
    ("armonk", "NY"): 10504,
    ("austin", "TX"): 78701,
    ("houston", "TX"): 77002,
    ("chicago", "IL"): 60601,
    ("redmond", "WA"): 98052,
    ("dublin", None): 20001,
    ("munich", None): 80333
}

def infer_country(city, state):
    if state in {"TX","NY","IL","WA","CA","MA","PA","FL"}:
        return "United States"
    if city == "Dublin":
        return "Ireland"
    if city == "Munich":
        return "Germany"
    return None

def infer_state(city, country):
    mapping = {
        "Austin": "TX", "Houston": "TX", "Chicago": "IL", "Redmond": "WA",
        "Armonk": "NY", "Munich": "BY", "Dublin": "DU"
    }
    if country == "United States" and city in mapping:
        return mapping[city]
    return None

def infer_city(state, country):
    mapping = {
        "TX": "Austin", "NY": "New York", "IL": "Chicago", "WA": "Redmond",
        "BY": "Munich", "DU": "Dublin"
    }
    if state in mapping:
        return mapping[state]
    if country == "Ireland":
        return "Dublin"
    if country == "Germany":
        return "Munich"
    return None

def infer_zip(city, state):
    for (c, s), z in ZIP_MAP.items():
        if city and str(city).lower() == c and (s is None or s == state):
            return z
    return None

# -------------------------------
# MAIN EXECUTION
# -------------------------------

if companies is not None:

    # 1) Basic cleaning
    for col in ["state", "country", "city", "zip_code", "address"]:
        if col in companies.columns:
            companies[col] = companies[col].apply(clean_field)

    # 2) Remove special chars
    if "city" in companies.columns:
        companies["city"] = companies["city"].apply(clean_special_chars_city)
    if "state" in companies.columns:
        companies["state"] = companies["state"].apply(clean_special_chars_state)
    if "country" in companies.columns:
        companies["country"] = companies["country"].apply(clean_special_chars_country)

    # 3) Normalize country & state
    if "country" in companies.columns:
        companies["country"] = companies["country"].apply(normalize_country)
    if "state" in companies.columns:
        companies["state"] = companies["state"].apply(normalize_state)

    # 4) Clean city outliers
    if "city" in companies.columns:
        companies.loc[companies["city"].isin(["Nan", "Remote", "Worldwide"]), "city"] = np.nan

    # 5) ZIP cleanup
    if "zip_code" in companies.columns:
        companies["zip_code"] = companies["zip_code"].astype(str).str.extract(r"(\d+)", expand=False)
        companies["zip_code"] = pd.to_numeric(companies["zip_code"], errors="coerce")

    # 6) Address cleanup
    if "address" in companies.columns:
        companies["address"] = companies["address"].apply(
            lambda x: np.nan
            if (not isinstance(x, str) or x.strip().lower() in {"nan", "-", ""})
            else x.title()
        )

    # --------------------------
    # GLOBAL FALLBACKS
    # --------------------------
    global_country = companies["country"].mode().iloc[0] if companies["country"].notna().sum() else "United States"
    # global_state will most likely be a 2–3 letter code from existing data
    global_state   = companies["state"].mode().iloc[0]   if companies["state"].notna().sum()   else "NY"
    global_city    = companies["city"].mode().iloc[0]    if companies["city"].notna().sum()    else "New York"
    global_zip     = int(companies["zip_code"].median()) if companies["zip_code"].notna().sum() else 99999
    global_address = "100 Main Street"

    # --------------------------
    # IMPUTE ROW BY ROW
    # --------------------------
    for idx, row in companies.iterrows():
        city    = row.get("city", np.nan)
        state   = row.get("state", np.nan)
        country = row.get("country", np.nan)
        zipc    = row.get("zip_code", np.nan)
        addr    = row.get("address", np.nan)

        # COUNTRY
        if pd.isna(country):
            country = infer_country(city, state) or global_country

        # STATE
        if pd.isna(state):
            if country == "United States":
                # try infer from city, else fallback to global_state (US code)
                state = infer_state(city, country) or global_state
            else:
                # non-US: build region abbrev from city or country
                if not pd.isna(city):
                    state = auto_abbrev(city)
                else:
                    state = auto_abbrev(country)

        # CITY
        if pd.isna(city):
            city = infer_city(state, country) or global_city

        # ZIP
        if pd.isna(zipc):
            zipc = infer_zip(city, state) or global_zip

        # ADDRESS
        if pd.isna(addr):
            addr = pick_address(city) if city else global_address

        companies.at[idx, "country"]  = country
        companies.at[idx, "state"]    = state
        companies.at[idx, "city"]     = city
        companies.at[idx, "zip_code"] = zipc
        companies.at[idx, "address"]  = addr

    # --------------------------
    # ZIP NORMALIZATION TO 5 DIGITS (AS STRING)
    # --------------------------
    companies["zip_code"] = companies["zip_code"].astype(str)
    companies["zip_code"] = companies["zip_code"].str.replace(r"\.0$", "", regex=True)
    companies["zip_code"] = companies["zip_code"].str.replace(r"[^0-9]", "", regex=True)
    companies["zip_code"] = companies["zip_code"].str.zfill(5)
    companies["zip_code"] = companies["zip_code"].str[:5]

    print("Location cleaning + state abbreviations + ZIP normalization + imputation complete.")

    # ================================================================
    # 3B. MEANINGFUL COMPANY SIZE ASSIGNMENT
    # ================================================================
    sizes = []

    for idx, row in companies.iterrows():
        name    = row.get("name", "")
        desc    = row.get("description", "")
        city    = row.get("city", "")
        country = row.get("country", "")

        # 1) NAME-based rules
        if contains(name, ["global", "international", "group", "holdings", "corp", "corporation", "enterprise", "industries"]):
            low, high = 5000, 200000
        elif contains(name, ["technologies", "technology", "systems", "solutions", "labs", "infrastructure"]):
            low, high = 1000, 150000
        elif contains(name, ["consulting", "advisors", "partners", "services", "studio", "llc", "clinic"]):
            low, high = 20, 3000
        else:
            low, high = 50, 20000

        # 2) DESCRIPTION-based modifiers
        if contains(desc, ["cloud", "ai", "ml", "platform", "software", "data", "infrastructure", "digital"]):
            low, high = max(low, 1000), max(high, 150000)
        if contains(desc, ["healthcare", "medical", "device", "clinician", "hospital"]):
            low, high = max(low, 800), max(high, 50000)
        if contains(desc, ["manufacturing", "industrial", "hardware", "engineering"]):
            low, high = max(low, 2000), max(high, 120000)
        if contains(desc, ["consulting", "services", "solutions", "advisory"]):
            low, high = max(low, 20), min(high, 8000)

        # 3) LOCATION-based modifiers
        big_cities = ["New York", "San Francisco", "Seattle", "Austin", "Chicago", "Houston", "Boston", "Los Angeles"]
        if any(c.lower() in str(city).lower() for c in big_cities):
            low = int(low * 1.1)
            high = int(high * 1.3)

        if country == "United States":
            high = int(high * 1.2)

        sizes.append(assign_size_from_range(low, high))

    companies["company_size"] = sizes

    # remove duplicates
    if "company_id" in companies.columns:
        before = len(companies)
        companies = companies.drop_duplicates(subset=["company_id"])
        print(f"Removed {before - len(companies)} duplicate rows.")

    print("Meaningful company size assignment complete. Shape:", companies.shape)

else:
    print("[WARN] companies.csv not found; skipping all cleaning.")


Location cleaning + state abbreviations + ZIP normalization + imputation complete.
Removed 0 duplicate rows.
Meaningful company size assignment complete. Shape: (24473, 10)


In [21]:
# --- 3C. CLEAN: skills & job_skills ---
import re

def clean_skill_abbrev(x):
    if not isinstance(x, str) or x.strip() == "":
        return None
    x = re.sub(r"[^A-Za-z]", "", x)  # remove non-letters
    if len(x) == 0:
        return None
    return x[:3].upper()

if skills is not None:
    for col in skills.columns:
        if skills[col].dtype == object:
            skills[col] = skills[col].apply(clean_text)
    skills = skills.drop_duplicates()
    skills["skill_abr"] = skills["skill_abr"].apply(clean_skill_abbrev)
    print('skills cleaned. Shape:', skills.shape)
else:
    print('[WARN] skills.csv not found; skipping skills cleaning.')

if job_skills is not None:
    # Coerce numeric ids
    for col in [c for c in ['job_id','skill_id'] if c in job_skills.columns]:
        job_skills[col] = pd.to_numeric(job_skills[col], errors='coerce')
    before = len(job_skills)
    job_skills = job_skills.dropna(subset=[c for c in ['job_id','skill_id'] if c in job_skills.columns])
    job_skills = job_skills.drop_duplicates()
    print(f"job_skills cleaned. Dropped {before - len(job_skills)} rows. Shape: {job_skills.shape}")
else:
    print('[WARN] job_skills.csv not found; skipping job_skills cleaning.')


skills cleaned. Shape: (35, 2)
job_skills cleaned. Dropped 0 rows. Shape: (213768, 2)


In [22]:
# --- 3D. CLEAN: industries & job_industries ---

# -------------------------------
# INDUSTRIES CLEANING
# -------------------------------
if industries is not None:

    for col in industries.columns:

        # Only process text columns
        if industries[col].dtype == object:
            # 1. Clean text
            industries[col] = industries[col].apply(clean_text)

            # 2. Convert all blank-like values to NaN
            industries[col] = industries[col].replace(
                {"", " ", "nan", "None", "-", None}, np.nan
            )

            # 3. Replace missing values with GLOBAL MODE
            if industries[col].isna().sum() > 0:
                if industries[col].mode().size > 0:
                    mode_value = industries[col].mode().iloc[0]
                else:
                    mode_value = "General"  # fallback
                industries[col] = industries[col].fillna(mode_value)

    # 4. Drop duplicates
    industries = industries.drop_duplicates()

    print(f"industries cleaned: no nulls, blanks replaced with global mode. Shape: {industries.shape}")

else:
    print("[WARN] industries.csv not found; skipping industries cleaning.")


# -------------------------------
# JOB_INDUSTRIES CLEANING
# -------------------------------
if job_industries is not None:

    # Convert job_id and industry_id to numeric
    for col in ['job_id', 'industry_id']:
        if col in job_industries.columns:
            job_industries[col] = pd.to_numeric(job_industries[col], errors='coerce')

    before = len(job_industries)

    # Remove rows missing required IDs
    job_industries = job_industries.dropna(subset=['job_id', 'industry_id'])

    # Remove duplicates
    job_industries = job_industries.drop_duplicates()

    print(f"job_industries cleaned. Dropped {before - len(job_industries)} rows. Shape: {job_industries.shape}")

else:
    print("[WARN] job_industries.csv not found; skipping job_industries cleaning.")


industries cleaned: no nulls, blanks replaced with global mode. Shape: (422, 2)
job_industries cleaned. Dropped 0 rows. Shape: (164808, 2)


In [23]:
# --- 3E. CLEAN: benefits ---
if benefits is not None:
    for col in benefits.columns:
        if benefits[col].dtype == object:
            benefits[col] = benefits[col].apply(clean_text)
    if 'inferred' in benefits.columns:
        # Convert to boolean if needed
        benefits['inferred'] = benefits['inferred'].astype(str).str.lower().isin(['true','1','yes','y','t'])
    benefits = benefits.drop_duplicates()
    print('benefits cleaned. Shape:', benefits.shape)
else:
    print('[WARN] benefits.csv not found; skipping benefits cleaning.')


benefits cleaned. Shape: (67943, 3)


In [24]:
# --- 3F. CLEAN: company_industries & company_specialities ---
if company_industries is not None:
    for col in [c for c in ['company_id','industry_id'] if c in company_industries.columns]:
        company_industries[col] = pd.to_numeric(company_industries[col], errors='coerce')
    company_industries = company_industries.dropna()
    company_industries = company_industries.drop_duplicates()
    print('company_industries cleaned. Shape:', company_industries.shape)
else:
    print('[WARN] company_industries.csv not found; skipping.')

if company_specialities is not None:
    for col in company_specialities.columns:
        if company_specialities[col].dtype == object:
            company_specialities[col] = company_specialities[col].apply(clean_text)
    company_specialities = company_specialities.drop_duplicates()
    print('company_specialities cleaned. Shape:', company_specialities.shape)
else:
    print('[WARN] company_specialities.csv not found; skipping.')


company_industries cleaned. Shape: (24375, 2)
company_specialities cleaned. Shape: (169209, 2)


In [25]:
# --- 3G. CLEAN: employee_counts ---
if employee_counts is not None:
    # Ensure numeric columns
    for col in [c for c in ['company_id','employee_count','follower_count','time_recorded'] if c in employee_counts.columns]:
        employee_counts[col] = pd.to_numeric(employee_counts[col], errors='coerce')
    employee_counts = employee_counts.dropna(subset=[c for c in ['company_id'] if c in employee_counts.columns])
    employee_counts = employee_counts.drop_duplicates()
    print('employee_counts cleaned. Shape:', employee_counts.shape)
else:
    print('[WARN] employee_counts.csv not found; skipping employee_counts cleaning.')


employee_counts cleaned. Shape: (35787, 4)


In [26]:
# # --- 3E. CLEAN + IMPUTE SALARY FIELDS ---

# ============================================================
# 1. ENSURE NUMERIC SALARY COLUMNS
# ============================================================
for col in ["min_salary", "med_salary", "max_salary"]:
    salaries[col] = pd.to_numeric(salaries[col], errors="coerce")


# ============================================================
# 2. PAY PERIOD → YEARLY CONVERSION
# ============================================================
CONVERSION = {
    "YEARLY": 1,
    "MONTHLY": 12,
    "WEEKLY": 52,
    "BIWEEKLY": 26,
    "HOURLY": 2080    # 40 hrs * 52 weeks
}

salaries["pay_period"] = salaries["pay_period"].astype(str).str.upper().str.strip()

# Unknown → YEARLY
salaries.loc[~salaries["pay_period"].isin(CONVERSION.keys()), "pay_period"] = "YEARLY"

# Convert all to yearly
for col in ["min_salary", "med_salary", "max_salary"]:
    salaries[col] = salaries[col] * salaries["pay_period"].map(CONVERSION)

# Enforce YEARLY everywhere
salaries["pay_period"] = "YEARLY"


# ============================================================
# 3. REMOVE UNREALISTIC SALARIES (< 10,000)
# ============================================================
THRESHOLD = 10000

for col in ["min_salary", "med_salary", "max_salary"]:
    salaries.loc[salaries[col] < THRESHOLD, col] = np.nan

print(f"✔ Replaced values < {THRESHOLD} with NaN.")


# ============================================================
# 4. SALARY IMPUTATION RULES
# ============================================================

# A: min + max → med
mask = salaries["med_salary"].isna() & salaries["min_salary"].notna() & salaries["max_salary"].notna()
salaries.loc[mask, "med_salary"] = (salaries.loc[mask, "min_salary"] + salaries.loc[mask, "max_salary"]) / 2

# B: only min → med, max
mask = salaries["med_salary"].isna() & salaries["min_salary"].notna() & salaries["max_salary"].isna()
salaries.loc[mask, "med_salary"] = salaries.loc[mask, "min_salary"] * 1.20
salaries.loc[mask, "max_salary"] = salaries.loc[mask, "min_salary"] * 1.50

# C: only max → med, min
mask = salaries["med_salary"].isna() & salaries["max_salary"].notna() & salaries["min_salary"].isna()
salaries.loc[mask, "med_salary"] = salaries.loc[mask, "max_salary"] * 0.80
salaries.loc[mask, "min_salary"] = salaries.loc[mask, "max_salary"] * 0.60

# D: only med → min, max
mask = salaries["med_salary"].notna() & salaries["min_salary"].isna() & salaries["max_salary"].isna()
salaries.loc[mask, "min_salary"] = salaries.loc[mask, "med_salary"] * 0.80
salaries.loc[mask, "max_salary"] = salaries.loc[mask, "med_salary"] * 1.20

# E: fully missing → use global medians
global_min = salaries["min_salary"].median()
global_med = salaries["med_salary"].median()
global_max = salaries["max_salary"].median()

mask = salaries[["min_salary", "med_salary", "max_salary"]].isna().all(axis=1)
salaries.loc[mask, "min_salary"] = global_min
salaries.loc[mask, "med_salary"] = global_med
salaries.loc[mask, "max_salary"] = global_max


# ============================================================
# 5. DROP UNUSED COLUMNS
# ============================================================
salaries = salaries.drop(columns=["pay_period", "compensation_type"], errors="ignore")

print("✔ All salaries converted to YEARLY, cleaned, imputed, and normalized.")


✔ Replaced values < 10000 with NaN.
✔ All salaries converted to YEARLY, cleaned, imputed, and normalized.


In [27]:
# --- 3I. CLEAN: postings (base cleanup) ---
if postings is not None:
    # Remove exact duplicates on typical identity fields if present
    key_cols = [c for c in ['title','company_id','location','listed_time','date_posted'] if c in postings.columns]
    if key_cols:
        before = len(postings)
        postings = postings.drop_duplicates(subset=key_cols)
        print(f"postings: removed {before - len(postings)} duplicate rows based on {key_cols}")

    # Drop rows missing essential fields
    essential = [c for c in ['title','company_id'] if c in postings.columns]
    if essential:
        before = len(postings)
        postings = postings.dropna(subset=essential)
        print(f"postings: dropped {before - len(postings)} rows missing essentials {essential}")

    # Text normalization for key text columns if present
    text_cols = [c for c in ['title','description','formatted_experience_level','location','work_type','formatted_work_type'] if c in postings.columns]
    for col in text_cols:
        postings[col] = postings[col].apply(clean_text)

    # Parse listed_time / date_posted / original_listed_time / closed_time / expiry as datetimes
    for time_col in ['listed_time','date_posted','original_listed_time','closed_time','expiry']:
        if time_col in postings.columns:
            postings[time_col] = parse_datetime_series(postings[time_col])

    # Derive year if listed_time available (might be dropped later)
    if 'listed_time' in postings.columns:
        postings['year'] = postings['listed_time'].dt.year

    # Derive remote flag
    if 'formatted_work_type' in postings.columns and 'remote_allowed' not in postings.columns:
        postings['remote_allowed'] = postings['formatted_work_type'].str.contains('remote', case=False, na=False)

    # Basic numeric sanitization for salary-like and volume fields if they exist
    for col in ['max_salary','med_salary','min_salary','applies','views','years_experience']:
        if col in postings.columns:
            postings[col] = pd.to_numeric(postings[col], errors='coerce')

    # Remove impossible salaries (<= 0)
    for col in ['max_salary','med_salary','min_salary']:
        if col in postings.columns:
            postings.loc[postings[col] <= 0, col] = np.nan

    # Optional: outlier trimming on salary columns
    for col in ['max_salary','med_salary','min_salary']:
        if col in postings.columns:
            postings[col] = remove_outliers_3sigma(postings[col])

    # --- REMOVE SALARY COLUMNS FROM POSTINGS FOR DB NORMALIZATION ---
    salary_cols = ['min_salary','med_salary','max_salary','currency','compensation_type']
    postings = postings.drop(columns=[c for c in salary_cols if c in postings.columns], errors='ignore')

    # --- REMOVE pay_period COLUMN COMPLETELY ---
    postings = postings.drop(columns=['pay_period'], errors='ignore')

    print('postings base cleaning complete. Shape:', postings.shape)

else:
    print('[WARN] postings.csv not found; skipping postings cleaning.')


# --- 3I.1 Additional postings imputations & enrichment (STRICT VERSION) ---

if postings is not None:

    import numpy as np

    # 1. LOCATION from COMPANIES (city, state, country)
    if (companies is not None) and ('company_id' in postings.columns) and ('company_id' in companies.columns):
        loc_cols = ['company_id','city','state','country']
        loc_cols = [c for c in loc_cols if c in companies.columns]
        comp_locs = companies[loc_cols].drop_duplicates(subset=['company_id'])

        postings = postings.merge(comp_locs, on='company_id', how='left', suffixes=('', '_company'))

        def build_location(row):
            parts = []
            if 'city' in row and pd.notna(row['city']):
                parts.append(str(row['city']))
            if 'state' in row and pd.notna(row['state']):
                parts.append(str(row['state']))
            # if 'country' in row and pd.notna(row['country']):
            #     parts.append(str(row['country']))
            if parts:
                return ', '.join(parts)
            # fallback to any existing location if no company info
            return row.get('location')

        postings['location'] = postings.apply(build_location, axis=1)
        postings['location'] = postings['location'].astype(str).str.replace(r'\s+,', ',', regex=True).str.replace(r',\s+', ', ', regex=True)
    else:
        # fallback: keep original but tidy a bit if companies not available
        if 'location' in postings.columns:
            postings['location'] = postings['location'].str.replace(' ,', ',', regex=False).str.replace(', ', ', ', regex=False)

    # 2. remote_allowed → "Yes"/"No"
    if 'remote_allowed' in postings.columns:
        postings['remote_allowed'] = postings['remote_allowed'].map({
            1: "Yes", "1": "Yes", True: "Yes", "true": "Yes", "True": "Yes", "yes": "Yes", "Yes": "Yes",
            0: "No", "0": "No", False: "No", "false": "No", "False": "No", "no": "No", "No": "No"
        })
        postings['remote_allowed'] = postings['remote_allowed'].fillna("No")

    # 3. views → fill nulls with avg(company_id, location), then global avg, then Int64
    if 'views' in postings.columns:
        postings['views'] = pd.to_numeric(postings['views'], errors='coerce')

        if {'company_id', 'location'}.issubset(postings.columns):
            postings['views'] = postings['views'].fillna(
                postings.groupby(['company_id','location'])['views'].transform('mean')
            )

        postings['views'] = postings['views'].fillna(postings['views'].mean())
        postings['views'] = postings['views'].round().astype('Int64')

    # 4. applies → random % of views (2–12%)
    if 'views' in postings.columns:
        pct = np.random.uniform(0.02, 0.12, len(postings))
        postings['applies'] = (postings['views'].astype(float) * pct).round().astype('Int64')

    # 5. original_listed_time → random dates between Nov 1 and Dec 15, 2025
    start = pd.Timestamp('2025-11-01')
    end   = pd.Timestamp('2025-12-15')
    n = len(postings)
    rand_days = np.random.randint(0, (end - start).days + 1, size=n)
    original_dt = start + pd.to_timedelta(rand_days, unit='D')
    postings['original_listed_time'] = original_dt

    # align listed_time and date_posted to original_listed_time if present
    if 'listed_time' in postings.columns:
        postings['listed_time'] = postings['original_listed_time']
    if 'date_posted' in postings.columns:
        postings['date_posted'] = postings['original_listed_time']

    # 6. expiry → 1–4 weeks after original_listed_time based on applies/views ratio
    if 'views' in postings.columns and 'applies' in postings.columns:
        views_safe = postings['views'].replace({0: np.nan}).astype(float)
        ratio = postings['applies'].astype(float) / views_safe
        ratio = ratio.fillna(ratio.mean())

        weeks = pd.Series(4, index=postings.index)  # default 4 weeks
        weeks[ratio >= 0.10] = 1
        weeks[(ratio >= 0.06) & (ratio < 0.10)] = 2
        weeks[(ratio >= 0.03) & (ratio < 0.06)] = 3
    else:
        weeks = pd.Series(3, index=postings.index)  # fallback 3 weeks if no data

    postings['expiry'] = postings['original_listed_time'] + pd.to_timedelta(weeks, unit='W')

    # 7. Format all date columns as mm-dd-yyyy strings (after all date logic is done)
    for time_col in ['listed_time','date_posted','original_listed_time','closed_time','expiry']:
        if time_col in postings.columns and np.issubdtype(postings[time_col].dtype, np.dtype('datetime64[ns]')):
            postings[time_col] = postings[time_col].dt.strftime('%m-%d-%Y')

    # 8. Remove extra columns you listed
    cols_to_remove = [
        'closed_time', 'formatted_experience_level', 'skills_desc', 'listed_time',
        'posting_domain', 'sponsored', 'work_type', 'year',
        'normalized_salary', 'zip_code', 'fips'
    ]
    postings = postings.drop(columns=[c for c in cols_to_remove if c in postings.columns], errors='ignore')

    print("Finished postings strict imputations + date & location enrichment. Shape:", postings.shape)

else:
    print("[WARN] postings is None; skipping strict postings enrichment.")


postings: removed 2323 duplicate rows based on ['title', 'company_id', 'location', 'listed_time']
postings: dropped 0 rows missing essentials ['title', 'company_id']
postings base cleaning complete. Shape: (117485, 26)
Finished postings strict imputations + date & location enrichment. Shape: (117485, 18)


In [28]:
DATE_COLUMNS = ['listed_time','date_posted','original_listed_time','closed_time','expiry'] # Columns to fix
INPUT_FORMAT = '%m-%d-%Y'   # The format of your dates (e.g., "04-11-2024")
# --- End Configuration ---
try:

    for col in DATE_COLUMNS:
        if col in postings.columns:
            print(f"Converting column: {col}...")
            # Convert text to datetime objects
            # errors='coerce' will turn any bad dates (like "ASAP") into NaT (NULL)
            temp_dates = pd.to_datetime(postings[col], format=INPUT_FORMAT, errors='coerce')
            
            # Convert back to a string, but in the new YYYY-MM-DD format
            # NaT (NULL) values will be saved as empty strings
            postings[col] = temp_dates.dt.strftime('%Y-%m-%d')
        else:
            print(f"Warning: Column '{col}' not found.")

except Exception as e:
    print(f"An error occurred: {e}")

Converting column: original_listed_time...
Converting column: expiry...


In [29]:
# --- 4. INTEGRATION & VALIDATION ---
qa_rows = []
def add_qa(metric, value):
    qa_rows.append({'metric': metric, 'value': value})

# Referential integrity checks
if (postings is not None) and (job_skills is not None) and ('job_id' in postings.columns) and ('job_id' in job_skills.columns):
    orphan_js = job_skills[~job_skills['job_id'].isin(postings['job_id'])]
    add_qa('job_skills.orphan_job_ids', len(orphan_js))
    print('Orphan job_skills rows (no matching job_id in postings):', len(orphan_js))

if (postings is not None) and (companies is not None) and ('company_id' in postings.columns) and ('company_id' in companies.columns):
    orphan_companies = postings[~postings['company_id'].isin(companies['company_id'])]
    add_qa('postings.orphan_company_ids', len(orphan_companies))
    print('Postings with orphan company_id:', len(orphan_companies))

if (job_industries is not None) and (industries is not None) and ('industry_id' in job_industries.columns) and ('industry_id' in industries.columns):
    orphan_ind = job_industries[~job_industries['industry_id'].isin(industries['industry_id'])]
    add_qa('job_industries.orphan_industry_ids', len(orphan_ind))
    print('Job_industries with orphan industry_id:', len(orphan_ind))

if (salaries is not None) and (postings is not None) and ('job_id' in salaries.columns) and ('job_id' in postings.columns):
    orphan_sal = salaries[~salaries['job_id'].isin(postings['job_id'])]
    add_qa('salaries.orphan_job_ids', len(orphan_sal))
    print('Salaries with orphan job_id:', len(orphan_sal))

# Basic uniqueness checks
def check_unique(df, cols, name):
    if df is None or not cols:  # Skip if no columns provided
        return
    dup = df.duplicated(subset=cols).sum()
    add_qa(f'{name}.duplicate_keys({"+".join(cols)})', int(dup))
    print(f'[UNIQUE CHECK] {name}: duplicates on {cols} =', int(dup))

check_unique(postings, ['job_id'] if postings is not None and 'job_id' in postings.columns else [], 'postings')
check_unique(companies, ['company_id'] if companies is not None and 'company_id' in companies.columns else [], 'companies')
check_unique(skills, ['skill_id'] if skills is not None and 'skill_id' in skills.columns else [], 'skills')
check_unique(industries, ['industry_id'] if industries is not None and 'industry_id' in industries.columns else [], 'industries')


Orphan job_skills rows (no matching job_id in postings): 17541
Postings with orphan company_id: 1
Job_industries with orphan industry_id: 0
Salaries with orphan job_id: 6017
[UNIQUE CHECK] postings: duplicates on ['job_id'] = 0
[UNIQUE CHECK] companies: duplicates on ['company_id'] = 0
[UNIQUE CHECK] industries: duplicates on ['industry_id'] = 0


In [30]:
# --- 5. EXPORT & QA LOGGING ---
datasets = {
    'postings_clean.csv': postings,
    'companies_clean.csv': companies,
    'skills_clean.csv': skills,
    'job_skills_clean.csv': job_skills,
    'industries_clean.csv': industries,
    'job_industries_clean.csv': job_industries,
    'benefits_clean.csv': benefits,
    'company_industries_clean.csv': company_industries,
    'company_specialities_clean.csv': company_specialities,
    'employee_counts_clean.csv': employee_counts,
    'salaries_clean.csv': salaries
}

export_rows = []
for name, df in datasets.items():
    if df is None:
        continue
    out_path = OUTPUT_DIR / name
    df.to_csv(out_path,date_format='%Y-%m-%d', index=False)
    export_rows.append({'file': name, 'rows': len(df), 'path': str(out_path)})
    print('Saved', name, '->', out_path)

# Cleaning log
cleaning_log = pd.DataFrame(export_rows)
cleaning_log['timestamp'] = datetime.now().isoformat()
cleaning_log.to_csv(OUTPUT_DIR / 'cleaning_log.csv', index=False)
print('Cleaning log saved ->', OUTPUT_DIR / 'cleaning_log.csv')

# QA validation report
qa_report = pd.DataFrame(qa_rows)
qa_report['timestamp'] = datetime.now().isoformat()
qa_report.to_csv(OUTPUT_DIR / 'qa_validation_report.csv', index=False)
print('QA validation report saved ->', OUTPUT_DIR / 'qa_validation_report.csv')


Saved postings_clean.csv -> cleaned\postings_clean.csv
Saved companies_clean.csv -> cleaned\companies_clean.csv
Saved skills_clean.csv -> cleaned\skills_clean.csv
Saved job_skills_clean.csv -> cleaned\job_skills_clean.csv
Saved industries_clean.csv -> cleaned\industries_clean.csv
Saved job_industries_clean.csv -> cleaned\job_industries_clean.csv
Saved benefits_clean.csv -> cleaned\benefits_clean.csv
Saved company_industries_clean.csv -> cleaned\company_industries_clean.csv
Saved company_specialities_clean.csv -> cleaned\company_specialities_clean.csv
Saved employee_counts_clean.csv -> cleaned\employee_counts_clean.csv
Saved salaries_clean.csv -> cleaned\salaries_clean.csv
Cleaning log saved -> cleaned\cleaning_log.csv
QA validation report saved -> cleaned\qa_validation_report.csv


In [31]:
# # -------------------------------------------------------------
# # --- 5. EXPORT & QA LOGGING (EXCEL VERSION, FULLY FIXED) ----
# # -------------------------------------------------------------

# import openpyxl
# from openpyxl.cell.cell import ILLEGAL_CHARACTERS_RE

# # Function to remove characters Excel does not allow
# def remove_illegal_excel_chars(df):
#     df = df.copy()
#     for col in df.columns:
#         if df[col].dtype == "object":
#             df[col] = df[col].astype(str).apply(
#                 lambda x: ILLEGAL_CHARACTERS_RE.sub("", x) if isinstance(x, str) else x
#             )
#     return df


# # All datasets to export
# datasets = {
#     'postings_clean.xlsx': postings,
#     'companies_clean.xlsx': companies,
#     'skills_clean.xlsx': skills,
#     'job_skills_clean.xlsx': job_skills,
#     'industries_clean.xlsx': industries,
#     'job_industries_clean.xlsx': job_industries,
#     'benefits_clean.xlsx': benefits,
#     'company_industries_clean.xlsx': company_industries,
#     'company_specialities_clean.xlsx': company_specialities,
#     'employee_counts_clean.xlsx': employee_counts,
#     'salaries_clean.xlsx': salaries
# }

# export_rows = []

# # Export loop
# for name, df in datasets.items():
#     if df is None:
#         continue

#     # CLEAN: Remove Excel-illegal characters from every text column
#     df = remove_illegal_excel_chars(df)

#     # Save Excel
#     out_path = OUTPUT_DIR / name
#     df.to_excel(out_path, index=False, engine='openpyxl')

#     export_rows.append({
#         'file': name,
#         'rows': len(df),
#         'path': str(out_path)
#     })

#     print('Saved', name, '->', out_path)


# # Cleaning log
# cleaning_log = pd.DataFrame(export_rows)
# cleaning_log['timestamp'] = datetime.now().isoformat()
# cleaning_log.to_excel(OUTPUT_DIR / 'cleaning_log.xlsx', index=False, engine='openpyxl')
# print('Cleaning log saved ->', OUTPUT_DIR / 'cleaning_log.xlsx')


# # QA validation report
# qa_report = pd.DataFrame(qa_rows)
# qa_report['timestamp'] = datetime.now().isoformat()
# qa_report.to_excel(OUTPUT_DIR / 'qa_validation_report.xlsx', index=False, engine='openpyxl')
# print('QA validation report saved ->', OUTPUT_DIR / 'qa_validation_report.xlsx')
